# Level-2 summarization experiments

The round-5 `level2` prompt produced weak parents on the real tree (index-style listing,
`node-29` labels, filler at the root; also run on Haiku while round 5 validated Sonnet).
This notebook does for level 2 what the original 6 rounds did for level 1: same
experiment discipline, real inputs, eyeball comparison.

**Inputs**: 3 real parents spanning the size range — smallest / medium / largest — with
their children's full level-1 summaries printed first, so the raw material is visible.

**Approaches** (× haiku/sonnet, skip-if-exists, everything persisted to
`level2_experiments` with prompt provenance):
1. `l2-baseline` — the round-5 prompt as-is (control), but with child TITLES in labels.
2. `l2-narrative` — synthesize ACROSS children: coherent prose about the subtree,
   organized by the content's own structure, never by child; carries up only
   subtree-relevant cautions.
3. `l2-hybrid` — narrative overview first, then a compact "Children:" routing list
   (one discriminative line per child).
4. `l2-routing` — the untested open item from the experiments: the compact routing
   flavor, brevity-first, for high tree levels.

In [1]:
import sqlite3
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import summarize_experiments as se
from IPython.display import display, Markdown, HTML

APP_DIR = Path.cwd()
db = sqlite3.connect(APP_DIR / ".hrono" / "hrono-v2.db")


def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def prompt_id(name, text):
    with db:
        db.execute("INSERT OR IGNORE INTO prompts (name, text) VALUES (?,?)", (name, text))
    return db.execute("SELECT id FROM prompts WHERE name = ? AND text = ?", (name, text)).fetchone()[0]


def claude_call(prompt, model):
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(se.call_claude, prompt, model).result()


db.execute("""
CREATE TABLE IF NOT EXISTS level2_experiments (
    node_id    INTEGER NOT NULL REFERENCES nodes(id),
    prompt_id  INTEGER NOT NULL REFERENCES prompts(id),
    name       TEXT    NOT NULL,   -- experiment name: <prompt>__<model>
    model      TEXT    NOT NULL,
    text       TEXT    NOT NULL,
    cost_usd   REAL,
    created_at TEXT,
    UNIQUE (node_id, name)
)""")


def children_of(node_id):
    return db.execute("""
        SELECT n.id, n.summary FROM edges e JOIN nodes n ON n.id = e.child_id
        WHERE e.parent_id = ? AND e.child_kind = 'node' ORDER BY n.id""", (node_id,)).fetchall()


def title_of(summary):
    return (summary or "").lstrip("# ").split("\n")[0][:90]


def render_children(node_id):
    return "\n\n".join(
        f"=== child node-{cid}: {title_of(s)} ===\n{s}" for cid, s in children_of(node_id))


l2 = db.execute("SELECT id, size FROM nodes WHERE level = 2 ORDER BY size").fetchall()
SAMPLES = [l2[0][0], l2[len(l2) // 2][0], l2[-1][0]]   # smallest / median / largest
print(f"level-2 parents: {l2}  -> samples {SAMPLES}")

level-2 parents: [(45, 3), (43, 4), (46, 6), (42, 7), (44, 9), (47, 12)]  -> samples [45, 42, 47]


## The raw material — what a level-2 prompt actually has to work with

Per sample parent: its current (bad) summary for reference, then every child's full
level-1 summary in a collapsible block. Read these first; the right prompt follows from
what is actually here.

In [2]:
import markdown as md_lib

for pid in SAMPLES:
    size, current = db.execute("SELECT size, summary FROM nodes WHERE id = ?", (pid,)).fetchone()
    kids = children_of(pid)
    html = [f"<h2>parent node {pid} — {size} children</h2>"]
    html.append(f"<details><summary><b>current summary (level2 prompt, haiku) — the baseline being replaced</b></summary>"
                f"<div style='background:#fff4f4;padding:8px'>{md_lib.markdown(current)}</div></details>")
    for cid, s in kids:
        html.append(f"<details><summary><b>child node-{cid}</b>: {title_of(s)}</summary>"
                    f"<div style='background:#fafbfc;padding:8px'>{md_lib.markdown(s)}</div></details>")
    display(HTML("".join(html)))

In [3]:
PROMPTS_L2 = {
    "l2-baseline": se.PROMPTS["level2"],

    "l2-narrative": """\
The texts below are the summaries of the child nodes of one parent node in a knowledge
tree. Write the parent node's summary as a coherent description of the whole subtree -
what a newcomer reads to understand this area of the system without opening any child.

Guidelines:
- Start by stating what the subtree is as a whole: find the real common point of the
  children, never just list them.
- Synthesize ACROSS children: describe the area's actual structure - how its parts
  relate, shared technologies, data flow, recurring patterns - organized by the
  content's own natural groupings. Do NOT structure the text by child node.
- Use concrete names (classes, files, config keys, domains) verbatim from the child
  summaries; never invent or generalize names.
- If the children's Caution notes contain hazards that matter for the whole subtree,
  end with a short Caution section keeping only the few most consequential ones; drop
  child-local trivia. Omit the section if nothing qualifies.

Output only the summary in markdown, nothing else.""",

    "l2-hybrid": """\
The texts below are the summaries of the child nodes of one parent node in a knowledge
tree. An AI agent reads ONLY the parent summary to decide whether this subtree holds the
answer to its question, and if so which child to descend into. Write the parent summary
in two parts:

1. A prose overview that synthesizes the subtree as a whole: what it is, how its parts
   actually relate (layers, data flow, shared technologies), organized by the content's
   own structure - never child-by-child. Use concrete names verbatim.
2. A final section "Children:" with exactly one line per child:
   `<child label> - what distinguishes it from the siblings`, with the concrete
   identifiers an agent would match a question against. Enough to pick the right child
   without opening it.

Never invent or generalize names. Output only the summary in markdown, nothing else.""",

    "l2-routing": """\
The texts below are summaries of the child nodes of one parent node in a knowledge tree.
Write the text that would let someone decide, without opening anything, whether what
they are looking for lives in this subtree - and if so, which child to open next: state
what the subtree is, what it contains, what kinds of questions it can answer, then one
short discriminative line per child (by its label). This summary sits at a high level of
the tree where brevity wins: be compact, every sentence should help the routing
decision. Use concrete names from the children; do not invent anything. Output only the
summary.""",
}

MODELS = ("haiku", "sonnet")
for pid in SAMPLES:
    content = render_children(pid)
    for pname, ptext in PROMPTS_L2.items():
        for model in MODELS:
            exp_name = f"{pname}__{model}"
            if db.execute("SELECT 1 FROM level2_experiments WHERE node_id = ? AND name = ?",
                          (pid, exp_name)).fetchone():
                continue
            result = claude_call(ptext + "\n\n" + content, model)
            with db:
                db.execute("INSERT INTO level2_experiments (node_id, prompt_id, name, model,"
                           " text, cost_usd, created_at) VALUES (?,?,?,?,?,?,?)",
                           (pid, prompt_id(pname, ptext), exp_name, model, result["text"],
                            result["cost_usd"], utcnow()))
            print(f"node {pid} {exp_name}: ${result['cost_usd']:.3f}")
total = db.execute("SELECT COUNT(*), SUM(cost_usd) FROM level2_experiments").fetchone()
print(f"{total[0]} experiment outputs, total ${total[1]:.2f}")

47 experiment outputs, total $5.88


## Results — side by side per parent, for eyeball review

Judge each against the raw material above: does it say what the subtree IS (not a list)?
Does it synthesize across children or just enumerate? Would it route an agent to the
right child? Is it an appropriate LENGTH for a high tree level?

In [4]:
for pid in SAMPLES:
    html = [f"<h2>parent node {pid}</h2>"]
    for pname in PROMPTS_L2:
        for model in MODELS:
            row = db.execute("SELECT text, cost_usd FROM level2_experiments"
                             " WHERE node_id = ? AND name = ?",
                             (pid, f"{pname}__{model}")).fetchone()
            if not row:
                continue
            text, cost = row
            html.append(
                f"<details><summary><b>{pname} / {model}</b> — {len(text)} chars, ${cost:.3f}</summary>"
                f"<div style='background:#fafbfc;padding:8px;border:1px solid #eee'>"
                f"{md_lib.markdown(text)}</div></details>")
    display(HTML("".join(html)))

## Round 2 — routing-free abstraction (the design correction)

Verdict on round 1: routing does NOT belong in the prompt — references are mechanical
(the level-1 principle), the LLM only abstracts. Narrative was the best shape but too
verbose and too child-by-child. Round 2: two routing-free prompts pushing for the
insight that only exists when the children are seen together — emergent rules, shared
conventions, the global picture — with hard brevity pressure. Models: sonnet and opus
only (haiku is for map-stage condensation, nothing else). The final node artifact =
summary + code-rendered children list, shown in the display cell.

In [5]:
PROMPTS_L2R2 = {
    "l2-abstract": """\
The texts below are the summaries of the child nodes of one parent node in a knowledge
tree. Write the parent node's summary - the abstraction of the whole subtree, one level
higher than any child.

Guidelines:
- Say what the subtree is as a whole and what role it plays in the system; find the
  real common point of the children, never just list them.
- Prefer the insights that only appear when the children are seen TOGETHER: shared
  conventions, recurring patterns, emergent rules, architectural decisions that span
  children. Do not restate what any single child already says.
- Implementation details belong to the children; mention a concrete name only when it
  is load-bearing for the global picture (a central class, a shared boundary, a
  system-wide config).
- If hazards from the children's Caution notes matter at subtree scale, keep the few
  most consequential ones in a short Caution section; omit it otherwise.
- Be substantially shorter than any child summary. Every sentence must earn its place
  at this level of abstraction.
- Never invent or generalize names.

Output only the summary in markdown, nothing else.""",

    "l2-abstract-rules": """\
The texts below are the summaries of the child nodes of one parent node in a knowledge
tree. Write the parent node's summary - the abstraction of the whole subtree, one level
higher than any child. The value of this summary is what can ONLY be seen when all the
children are viewed together; anything a single child already says does not belong here.

Structure the output as:
1. A short opening: what this subtree is as a whole and its role in the system.
2. "Conventions & rules" - the emergent laws that hold across the children: shared
   stack and patterns, naming/layout conventions, invariants, boundaries - the things
   someone must know before touching ANYTHING in this subtree. Concrete names only
   where load-bearing.
3. "Caution" - only hazards that matter at subtree scale, from the children's caution
   notes; omit the section if none qualify.

Be substantially shorter than any child summary. Never invent or generalize names.
Output only the summary in markdown, nothing else.""",
}

R2_MODELS = ("sonnet", "opus")
for pid in SAMPLES:
    content = render_children(pid)
    for pname, ptext in PROMPTS_L2R2.items():
        for model in R2_MODELS:
            exp_name = f"{pname}__{model}"
            if db.execute("SELECT 1 FROM level2_experiments WHERE node_id = ? AND name = ?",
                          (pid, exp_name)).fetchone():
                continue
            result = claude_call(ptext + "\n\n" + content, model)
            with db:
                db.execute("INSERT INTO level2_experiments (node_id, prompt_id, name, model,"
                           " text, cost_usd, created_at) VALUES (?,?,?,?,?,?,?)",
                           (pid, prompt_id(pname, ptext), exp_name, model, result["text"],
                            result["cost_usd"], utcnow()))
            print(f"node {pid} {exp_name}: ${result['cost_usd']:.3f}")

## Round-2 results — rendered as the FINAL node artifact

Summary (LLM) + children list (mechanical, from edges + child titles) — what the tree
will actually store and show. Compare against the round-1 narrative above.

In [6]:
def mechanical_children(pid):
    lines = [f"- **node-{cid}** — {title_of(s)}" for cid, s in children_of(pid)]
    return "**Children (mechanical):**\n" + "\n".join(lines)


for pid in SAMPLES:
    html = [f"<h2>parent node {pid} — round 2</h2>"]
    for pname in PROMPTS_L2R2:
        for model in R2_MODELS:
            row = db.execute("SELECT text, cost_usd FROM level2_experiments"
                             " WHERE node_id = ? AND name = ?",
                             (pid, f"{pname}__{model}")).fetchone()
            if not row:
                continue
            text, cost = row
            html.append(
                f"<details><summary><b>{pname} / {model}</b> — {len(text)} chars, ${cost:.3f}</summary>"
                f"<div style='background:#f6fff6;padding:8px;border:1px solid #dde'>"
                f"{md_lib.markdown(text)}</div></details>")
    display(HTML("".join(html)))

## Round 3 — intent over structure

Round-2 verdict: `l2-abstract-rules` content is right (emergent rules + enough names to
anchor search; plain `l2-abstract` too name-sparse) but the mandated section skeleton is
wrong — level 1 already taught that structure must fit the content, not be imposed.
Round 3 keeps the intent, drops the skeleton. Display shows the LLM output ONLY (the
mechanical children list confused more than it demonstrated — in the real tree children
are visible as nested nodes anyway).

In [7]:
PROMPTS_L2R3 = {
    "l2-abstract-v3": """\
The texts below are the summaries of the child nodes of one parent node in a knowledge
tree. Write the parent node's summary - the abstraction of the whole subtree, one level
higher than any child.

What the reader must come away with:
- what this subtree is as a whole and its role in the system - the real common point of
  the children, never a list of them;
- the insight that only exists when the children are seen TOGETHER: shared conventions,
  recurring patterns, emergent rules, invariants, architectural decisions that span
  children. This is the summary's main value; anything a single child already says does
  not belong here;
- the main components of the subtree, by name - enough concrete anchors for further
  search and referencing, without the implementation detail that belongs to the
  children;
- the few hazards from the children's caution notes that matter at subtree scale, if
  there are any.

Choose whatever structure fits this content; do not force one. Be substantially shorter
than any child summary. Never invent or generalize names. Output only the summary in
markdown, nothing else.""",
}

for pid in SAMPLES:
    content = render_children(pid)
    for pname, ptext in PROMPTS_L2R3.items():
        for model in ("sonnet", "opus"):
            exp_name = f"{pname}__{model}"
            if db.execute("SELECT 1 FROM level2_experiments WHERE node_id = ? AND name = ?",
                          (pid, exp_name)).fetchone():
                continue
            result = claude_call(ptext + "\n\n" + content, model)
            with db:
                db.execute("INSERT INTO level2_experiments (node_id, prompt_id, name, model,"
                           " text, cost_usd, created_at) VALUES (?,?,?,?,?,?,?)",
                           (pid, prompt_id(pname, ptext), exp_name, model, result["text"],
                            result["cost_usd"], utcnow()))
            print(f"node {pid} {exp_name}: ${result['cost_usd']:.3f}")

for pid in SAMPLES:
    html = [f"<h2>parent node {pid} — round 3</h2>"]
    for model in ("sonnet", "opus"):
        row = db.execute("SELECT text, cost_usd FROM level2_experiments"
                         " WHERE node_id = ? AND name = ?",
                         (pid, f"l2-abstract-v3__{model}")).fetchone()
        if row:
            html.append(
                f"<details open><summary><b>l2-abstract-v3 / {model}</b> — {len(row[0])} chars,"
                f" ${row[1]:.3f}</summary><div style='background:#f6fff6;padding:8px;"
                f"border:1px solid #dde'>{md_lib.markdown(row[0])}</div></details>")
    display(HTML("".join(html)))

## Round 4 — pure summarization: no tree vocabulary anywhere

The LLM should not know it is inside a tree. Round-3 outputs cited "node-22" because
the INPUT labels leaked ids; the prompt spoke of nodes/children/subtrees. Round 4:
children rendered as plain titled texts, prompt stripped to pure summarization language.
Model: sonnet (adopted — opus's marginal edge doesn't justify 2x cost on an operation
that reruns on every membership change; per-node escalation stays possible via
summary_model).

In [8]:
def render_children_pure(node_id):
    return "\n\n".join(
        f"=== {title_of(s)} ===\n{s}" for _, s in children_of(node_id))


PROMPT_L2R4 = """\
Each text below is the summary of one group of related content. Together the groups
form one larger area. Write the summary of that whole area - one level of abstraction
higher than the texts.

The reader must come away with:
- what the area is as a whole;
- what only becomes visible when the groups are seen together: shared conventions,
  recurring patterns, emergent rules, invariants, cross-cutting decisions. This is the
  summary's main value; do not restate what a single text already says;
- the main components, by name - enough anchors for further search and reference,
  without deeper implementation detail;
- the few hazards from the texts' caution notes that matter for the whole area, if
  there are any.

Choose whatever structure fits the content. Be substantially shorter than any of the
texts. Never invent or generalize names. Output only the summary in markdown."""

for pid in SAMPLES:
    exp_name = "l2-pure-v4__sonnet"
    if not db.execute("SELECT 1 FROM level2_experiments WHERE node_id = ? AND name = ?",
                      (pid, exp_name)).fetchone():
        result = claude_call(PROMPT_L2R4 + "\n\n" + render_children_pure(pid), "sonnet")
        with db:
            db.execute("INSERT INTO level2_experiments (node_id, prompt_id, name, model,"
                       " text, cost_usd, created_at) VALUES (?,?,?,?,?,?,?)",
                       (pid, prompt_id("l2-pure-v4", PROMPT_L2R4), exp_name, "sonnet",
                        result["text"], result["cost_usd"], utcnow()))
        print(f"node {pid} {exp_name}: ${result['cost_usd']:.3f}")

for pid in SAMPLES:
    row = db.execute("SELECT text, cost_usd FROM level2_experiments WHERE node_id = ?"
                     " AND name = 'l2-pure-v4__sonnet'", (pid,)).fetchone()
    display(HTML(f"<h2>parent node {pid} — round 4 (l2-pure-v4 / sonnet, {len(row[0])} chars,"
                 f" ${row[1]:.3f})</h2><div style='background:#f6fff6;padding:8px;"
                 f"border:1px solid #dde'>{md_lib.markdown(row[0])}</div>"))

## Round 5 — content requirements are not an outline (single-cluster iteration)

Round-4 outputs echoed the prompt's bullets back as vague section titles ("shared
conventions", "main components", "hazards"). The requirements describe what to CONVEY,
not the form. Round 5 states that explicitly and demands the structure come from the
material's own concepts. Also: iterate on ONE cluster (the hardest, node 47 with 12
children) until it's beautiful, then validate on the other two — token economy.

In [9]:
ITER_NODE = 47   # largest parent — the stress test

PROMPT_L2R5 = """\
Each text below is the summary of one group of related content. Together the groups
form one larger area. Write the summary of that whole area - one level of abstraction
higher than the texts.

The following are content requirements: they say what the summary must convey, never
sections to create or headings to use. Organize the text around the area's OWN concepts
and names, the way a well-written overview of this specific material would be - its
structure should be unrecognizable from this list:
- what the area is as a whole;
- what only becomes visible when the groups are seen together: shared conventions,
  recurring patterns, emergent rules, invariants, cross-cutting decisions - woven into
  the description of the area itself, and never restating what a single text already
  says;
- the handful of components someone would search for first, by name, as anchors woven
  into the text - not a listing, and no deeper implementation detail;
- hazards from the texts' caution notes only where they matter for the whole area.

Be substantially shorter than any of the texts; the length must not grow with the
number of input texts. Never invent or generalize names. Output only the summary in
markdown."""

exp_name = "l2-pure-v5__sonnet"
if not db.execute("SELECT 1 FROM level2_experiments WHERE node_id = ? AND name = ?",
                  (ITER_NODE, exp_name)).fetchone():
    result = claude_call(PROMPT_L2R5 + "\n\n" + render_children_pure(ITER_NODE), "sonnet")
    with db:
        db.execute("INSERT INTO level2_experiments (node_id, prompt_id, name, model, text,"
                   " cost_usd, created_at) VALUES (?,?,?,?,?,?,?)",
                   (ITER_NODE, prompt_id("l2-pure-v5", PROMPT_L2R5), exp_name, "sonnet",
                    result["text"], result["cost_usd"], utcnow()))
    print(f"node {ITER_NODE} {exp_name}: ${result['cost_usd']:.3f}")

row = db.execute("SELECT text, cost_usd FROM level2_experiments WHERE node_id = ?"
                 " AND name = ?", (ITER_NODE, exp_name)).fetchone()
display(HTML(f"<h2>node {ITER_NODE} — round 5 ({len(row[0])} chars, ${row[1]:.3f})</h2>"
             f"<div style='background:#f6fff6;padding:8px;border:1px solid #dde'>"
             f"{md_lib.markdown(row[0])}</div>"))

## Round 6 — complete freedom of form

Clarification: the ban is on REQUIREMENT-derived headers, not on structure. The model
gets full formatting freedom (headings, lists, tables, any markdown) — with any headings
named from the content's own concepts. Only the intent is prescribed.

In [10]:
PROMPT_L2R6 = """\
Each text below is the summary of one group of related content. Together the groups
form one larger area. Write the summary of that whole area - one level of abstraction
higher than the texts.

These are content requirements only - they say what the summary must convey, and they
must not be recognizable in its form:
- what the area is as a whole;
- what only becomes visible when the groups are seen together: shared conventions,
  recurring patterns, emergent rules, invariants, cross-cutting decisions - never
  restating what a single text already says;
- the handful of components someone would search for first, by name, as anchors for
  further reference;
- hazards from the texts' caution notes, where they matter for the whole area.

The form is completely free: use whatever markdown serves this particular content best -
headings, lists, tables, prose, or any mix. If you use headings, name them after the
content's own concepts, never after the requirements above. Be substantially shorter
than any of the texts; the length must not grow with the number of input texts. Never
invent or generalize names. Output only the summary in markdown."""

exp_name = "l2-pure-v6__sonnet"
if not db.execute("SELECT 1 FROM level2_experiments WHERE node_id = ? AND name = ?",
                  (ITER_NODE, exp_name)).fetchone():
    result = claude_call(PROMPT_L2R6 + "\n\n" + render_children_pure(ITER_NODE), "sonnet")
    with db:
        db.execute("INSERT INTO level2_experiments (node_id, prompt_id, name, model, text,"
                   " cost_usd, created_at) VALUES (?,?,?,?,?,?,?)",
                   (ITER_NODE, prompt_id("l2-pure-v6", PROMPT_L2R6), exp_name, "sonnet",
                    result["text"], result["cost_usd"], utcnow()))
    print(f"node {ITER_NODE} {exp_name}: ${result['cost_usd']:.3f}")

row = db.execute("SELECT text, cost_usd FROM level2_experiments WHERE node_id = ?"
                 " AND name = ?", (ITER_NODE, exp_name)).fetchone()
display(HTML(f"<h2>node {ITER_NODE} — round 6 ({len(row[0])} chars, ${row[1]:.3f})</h2>"
             f"<div style='background:#f6fff6;padding:8px;border:1px solid #dde'>"
             f"{md_lib.markdown(row[0])}</div>"))

## Round 7 — polishing v6: echo removal and size control

v6 is the working best. Three probes on top of it (single cluster, sonnet):
- `v7a-noecho` — only change: "as anchors" wording dropped ("the components someone
  would search for first"), testing whether the 'Anchor components' header echo dies.
- `v7b-sized` — v7a + explicit size target (~500 words), knowing round 1 found
  prompt-level budgets get ignored — testing whether that still holds here.
- `v7c-editor` — v7a + self-editing pressure ("draft, then strike every word that can
  go without losing meaning") instead of a numeric target.

In [11]:
V6_CORE = """\
Each text below is the summary of one group of related content. Together the groups
form one larger area. Write the summary of that whole area - one level of abstraction
higher than the texts.

These are content requirements only - they say what the summary must convey, and they
must not be recognizable in its form:
- what the area is as a whole;
- what only becomes visible when the groups are seen together: shared conventions,
  recurring patterns, emergent rules, invariants, cross-cutting decisions - never
  restating what a single text already says;
- the components someone would search for first, by name;
- hazards from the texts' caution notes, where they matter for the whole area.

The form is completely free: use whatever markdown serves this particular content best -
headings, lists, tables, prose, or any mix. If you use headings, name them after the
content's own concepts, never after the requirements above."""

TAIL_COMMON = """ Never invent or generalize names. Output only the summary in markdown."""

R7 = {
    "v7a-noecho": V6_CORE + """ Be substantially shorter
than any of the texts; the length must not grow with the number of input texts.""" + TAIL_COMMON,
    "v7b-sized": V6_CORE + """ Aim for roughly 500 words;
compress by tightening the wording, never by dropping meaning.""" + TAIL_COMMON,
    "v7c-editor": V6_CORE + """ First draft mentally, then
write the compressed version: strike every word that can go without losing meaning -
wordiness is a defect at this level of abstraction.""" + TAIL_COMMON,
}

for pname, ptext in R7.items():
    exp_name = f"{pname}__sonnet"
    if not db.execute("SELECT 1 FROM level2_experiments WHERE node_id = ? AND name = ?",
                      (ITER_NODE, exp_name)).fetchone():
        result = claude_call(ptext + "\n\n" + render_children_pure(ITER_NODE), "sonnet")
        with db:
            db.execute("INSERT INTO level2_experiments (node_id, prompt_id, name, model, text,"
                       " cost_usd, created_at) VALUES (?,?,?,?,?,?,?)",
                       (ITER_NODE, prompt_id(pname, ptext), exp_name, "sonnet",
                        result["text"], result["cost_usd"], utcnow()))
        print(f"{exp_name}: ${result['cost_usd']:.3f}")

for pname in R7:
    row = db.execute("SELECT text, cost_usd FROM level2_experiments WHERE node_id = ?"
                     " AND name = ?", (ITER_NODE, f"{pname}__sonnet")).fetchone()
    display(HTML(f"<h2>node {ITER_NODE} — {pname} ({len(row[0])} chars, ${row[1]:.3f})</h2>"
                 f"<div style='background:#f6fff6;padding:8px;border:1px solid #dde'>"
                 f"{md_lib.markdown(row[0])}</div>"))

v7a-noecho__sonnet: $0.177


v7b-sized__sonnet: $0.172


v7c-editor__sonnet: $0.186


## Observations

(To fill after review: which prompt, which model, and whether the root wants the same
treatment or its own — then tree-build.ipynb adopts the winner and regenerates level 2+.)